# Redraw figures — Cox's Bazar–Teknaf oceanographic forcing

Redraws all figures from the results already produced by `ocean_forcing.ipynb` (no ocean data is downloaded or recomputed),
using the exact hotspot polygons in `hotspots.geojson`, the study-area outline in `study_area.geojson`
and a high-resolution GSHHS coastline.

1. **Runtime > Run all.** Needs `ocean_outputs.zip`, `study_area.geojson` and `hotspots.geojson`
   (upload them when asked if they are not in the working folder).
2. Outputs go to `./figures_v2` and are downloaded as `figures_v2.zip` at the end.
3. Send the zip and the printed `SUMMARY v2` to Claude.

In [ ]:
!pip -q install cartopy geopandas

In [ ]:
# SETUP: locate / upload the input files, unzip the results, create the output folder
import os, io, json, shutil, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy import stats
from scipy.interpolate import griddata
from IPython.display import display
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

ZIP_NAME, STUDY_NAME, HOT_NAME = "ocean_outputs.zip", "study_area.geojson", "hotspots.geojson"
DATA_DIR, FIG_DIR = "./ocean_outputs", "./figures_v2"

for fname in (ZIP_NAME, STUDY_NAME, HOT_NAME):
    if not os.path.exists(fname):
        print(f"{fname} not found in {os.getcwd()} -> please upload it")
        try:
            from google.colab import files
            files.upload()                      # saves the chosen files into the working directory
        except ImportError:
            pass
    if not os.path.exists(fname):
        raise FileNotFoundError(f"{fname} is required (put it in {os.getcwd()})")

with zipfile.ZipFile(ZIP_NAME) as zf:
    zf.extractall(DATA_DIR)
os.makedirs(FIG_DIR, exist_ok=True)
print("results unzipped to", DATA_DIR, "| outputs go to", FIG_DIR)


def read_csv(name):
    """Read one results CSV from the unzipped folder (searches sub-folders too)."""
    for root, _, names in os.walk(DATA_DIR):
        if name in names:
            return pd.read_csv(os.path.join(root, name))
    raise FileNotFoundError(name)


plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "figure.dpi": 110,
                     "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
PC = ccrs.PlateCarree()


def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=300)
    plt.show()
    print("saved", name)


def save_table(df, name):
    df.to_csv(os.path.join(FIG_DIR, name), index=False)
    print("saved", name)

In [ ]:
# STEP 1: exact hotspot positions from hotspots.geojson
ZONE_INFO = {   # label: (full name, kind, EPR m/yr, approximate lat, approximate lon used in the original analysis)
    "A1": ("A1 Nazirartek",                    "Accretion", +13.74, 21.468, 91.958),
    "E3": ("E3 Flight Deck BAF / Somiti Para", "Erosion",    -4.80, 21.445, 91.962),
    "A3": ("A3 Sonar Para / BORI Beach",       "Accretion",  +5.59, 21.330, 92.020),
    "A2": ("A2 Reju Khal",                     "Accretion",  +7.88, 21.262, 92.045),
    "E2": ("E2 Sabrang",                       "Erosion",    -4.85, 20.805, 92.285),
    "E1": ("E1 Dakhinpara",                    "Erosion",   -24.46, 20.762, 92.318),
}                                                   # north -> south along the coast
ORDER = list(ZONE_INFO)
KEYWORDS = {"A1": ["nazirartek"], "E3": ["somiti", "flight"], "A3": ["sonar", "bori"],
            "A2": ["reju"], "E2": ["sabrang"], "E1": ["dakhinpara"]}
R_EARTH = 6371.0


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))


hot = gpd.read_file(HOT_NAME)
if hot.crs is None:
    hot = hot.set_crs(32646)
hot = hot.to_crs(4326)
print("hotspots.geojson columns:", [c for c in hot.columns if c != "geometry"])
print("geometry types:", hot.geom_type.value_counts().to_dict(), "| features:", len(hot))
display(hot.drop(columns="geometry"))


def match_zones(gdf):
    """Return {label: row index} using any attribute text; None if not a confident one-to-one match."""
    import re
    found = {}
    for idx, row in gdf.drop(columns="geometry").iterrows():
        text = " | ".join(str(v).lower() for v in row.values if v is not None and str(v) != "nan")
        hits = [lab for lab, kws in KEYWORDS.items()
                if any(k in text for k in kws) or re.search(rf"(?<![a-z0-9]){lab.lower()}(?![a-z0-9])", text)]
        if len(hits) != 1:
            return None
        found.setdefault(hits[0], []).append(idx)
    if set(found) == set(ORDER) and all(len(v) == 1 for v in found.values()):
        return {lab: v[0] for lab, v in found.items()}
    return None


USED_APPROX = False
mapping = match_zones(hot)
if mapping is not None:
    MATCH_METHOD = "attribute match (keywords / zone codes)"
    zone_geoms = {lab: hot.loc[i, "geometry"] for lab, i in mapping.items()}
    for lab, i in mapping.items():          # sanity check against the expected erosion / accretion type
        for col in hot.columns:
            if str(hot.loc[i, col]).lower() in ("erosion", "accretion") and \
                    str(hot.loc[i, col]).lower() != ZONE_INFO[lab][1].lower():
                print(f"WARNING: {lab} is '{hot.loc[i, col]}' in the file but {ZONE_INFO[lab][1]} expected")
else:
    # no usable attribute: try six features, or six groups after dissolving by an id field, ordered N -> S
    groups = None
    if len(hot) == 6:
        groups = hot.geometry.reset_index(drop=True)
    else:
        for col in [c for c in hot.columns if c.lower() in ("fid", "id", "zone", "name", "label")]:
            d = hot.dissolve(by=col)
            if len(d) == 6:
                groups = d.geometry.reset_index(drop=True)
                break
    if groups is not None:
        lat_c = groups.to_crs(32646).centroid.to_crs(4326).y
        ranked = groups.loc[lat_c.sort_values(ascending=False).index]
        zone_geoms = dict(zip(ORDER, ranked.values))
        MATCH_METHOD = "NO usable attribute: assigned north -> south by latitude"
        print("WARNING: could not match attributes; zones assigned by latitude order")
    else:
        USED_APPROX = True
        MATCH_METHOD = "FAILED: fell back to approximate coordinates"
        print("WARNING: hotspots.geojson could not be matched confidently to six zones -> using the approximate "
              "coordinates from the original analysis (no polygons).")
print("match method:", MATCH_METHOD)

rows = []
for lab in ORDER:
    name, kind, epr, lat0, lon0 = ZONE_INFO[lab]
    if USED_APPROX:
        lat, lon = lat0, lon0
    else:   # centroid in UTM 46N (metres), converted back to lon/lat
        c = gpd.GeoSeries([zone_geoms[lab]], crs=4326).to_crs(32646).centroid.to_crs(4326).iloc[0]
        lat, lon = c.y, c.x
    rows.append(dict(zone=name, label=lab, kind=kind, epr=epr, lat=lat, lon=lon,
                     old_lat=lat0, old_lon=lon0, shift_km=float(haversine_km(lat0, lon0, lat, lon))))
POS = pd.DataFrame(rows)
display(POS[["label", "zone", "old_lat", "old_lon", "lat", "lon", "shift_km"]].round(4))
save_table(POS[["zone", "label", "kind", "epr", "lat", "lon", "shift_km"]], "hotspot_positions.csv")

ZONE_GEOMS = {} if USED_APPROX else zone_geoms     # polygons / lines for the maps (EPSG:4326)
study = gpd.read_file(STUDY_NAME)
study = study.set_crs(4326) if study.crs is None else study.to_crs(4326)

In [ ]:
# STEP 2: recompute the hotspot linkage table from the CSVs already in the zip
HYCOM = {"sw": read_csv("hycom_sw.csv"), "ne": read_csv("hycom_ne.csv"), "ann": read_csv("hycom_ann.csv")}
WAVES = read_csv("era5_wave_stats.csv")
SEASON_KEY = {"sw": "SW", "ne": "NE", "ann": "Annual"}              # matched against the season text
WAVE_SEASON = {k: WAVES[WAVES["season"].str.contains(t, case=False)].reset_index(drop=True)
               for k, t in SEASON_KEY.items()}
print("columns in era5_wave_stats.csv:", list(WAVES.columns))


def destination(lat, lon, az_deg, dist_km):
    """Spherical great-circle destination point."""
    p1, l1, az, d = np.radians(lat), np.radians(lon), np.radians(az_deg), dist_km / R_EARTH
    p2 = np.arcsin(np.sin(p1) * np.cos(d) + np.cos(p1) * np.sin(d) * np.cos(az))
    l2 = l1 + np.arctan2(np.sin(az) * np.sin(d) * np.cos(p1), np.cos(d) - np.sin(p1) * np.sin(p2))
    return float(np.degrees(p2)), float(np.degrees(l2))


def nearest(df, lat, lon, lat_col="lat", lon_col="lon"):
    d = haversine_km(lat, lon, df[lat_col].values, df[lon_col].values)
    i = int(np.argmin(d))
    return df.iloc[i], float(d[i])


rows = []
for _, z in POS.iterrows():
    off_lat, off_lon = destination(z.lat, z.lon, 245.0, 10.0)      # 10 km offshore (WSW)
    row = dict(zone=z.zone, label=z.label, kind=z.kind, epr=z.epr)
    for k in ("sw", "ne", "ann"):
        h, h_km = nearest(HYCOM[k].dropna(subset=["speed"]), off_lat, off_lon)
        w, w_km = nearest(WAVE_SEASON[k], off_lat, off_lon, "grid_lat", "grid_lon")
        row.update({f"cur_speed_{k}": h["speed"], f"cur_along_{k}": h["along"], f"sst_{k}": h["sst"],
                    f"hs_mean_{k}": w["hs_mean"], f"hs_p90_{k}": w["hs_p90"], f"Pl_net_{k}": w["Pl_net"],
                    f"wave_dir_from_{k}": w["dir_from"], f"era5_cell_{k}": w["cell"],
                    f"hycom_dist_km_{k}": h_km, f"era5_dist_km_{k}": w_km})
    rows.append(row)
LINK = pd.DataFrame(rows)
save_table(LINK, "hotspot_forcing_v2.csv")
display(LINK[["label", "kind", "epr", "cur_along_sw", "sst_ann", "hs_mean_sw", "hs_mean_ann", "Pl_net_ann",
              "era5_cell_ann", "hycom_dist_km_ann", "era5_dist_km_ann"]].round(3))

# comparison with the original table
old = read_csv("hotspot_forcing.csv")
changed = []
for col in old.columns:
    if col in ("zone", "label", "kind") or col not in LINK.columns:
        continue
    for i, lab in enumerate(old["label"]):
        a, b = old.loc[i, col], LINK.set_index("label").loc[lab, col]
        same = (a == b) if isinstance(a, str) else bool(np.isclose(a, b, rtol=1e-6, atol=1e-9))
        if not same:
            changed.append(dict(label=lab, column=col, old=a, new=b))
CHANGES = pd.DataFrame(changed, columns=["label", "column", "old", "new"])
save_table(CHANGES, "hotspot_changes_v2.csv")
if CHANGES.empty:
    print("Comparison: every value in hotspot_forcing_v2.csv equals the original table.")
else:
    value_cols = CHANGES[~CHANGES["column"].str.contains("dist_km")]
    print(f"Comparison: {len(CHANGES)} values differ from the original table "
          f"({len(value_cols)} forcing values, {len(CHANGES) - len(value_cols)} distances).")
    print("Forcing values changed per zone:", value_cols.groupby("label").size().to_dict())
    display(value_cols.head(30))

# Spearman correlation of EPR (6 zones: purely exploratory)
corr = []
for k in ("sw", "ne", "ann"):
    for var in ["cur_speed", "cur_along", "sst", "hs_mean", "hs_p90", "Pl_net"]:
        col = f"{var}_{k}"
        n_distinct = int(LINK[col].round(6).nunique())
        if n_distinct >= 4:
            rho, p = stats.spearmanr(LINK["epr"], LINK[col])
            corr.append(dict(variable=col, n_distinct=n_distinct, rho=rho, p=p, note="tested"))
        else:
            corr.append(dict(variable=col, n_distinct=n_distinct, rho=np.nan, p=np.nan,
                             note="not tested – zones share grid cells"))
CORR = pd.DataFrame(corr)
save_table(CORR, "hotspot_correlations_v2.csv")
display(CORR.round(3))

In [ ]:
# STEP 3: shared map style (coastline, study area, hotspots, labels)
def probe(feature):
    """Force the coastline download/drawing once so that a failure shows up here, not later."""
    fig = plt.figure(figsize=(2, 2))
    ax = fig.add_subplot(projection=PC)
    ax.set_extent([91.8, 91.9, 21.0, 21.1], crs=PC)
    ax.add_feature(feature)
    fig.canvas.draw()
    plt.close(fig)


LAND_FC, LAND_EC = "#efe6cf", "#4a3b2a"
COAST = None
for desc, make in [("GSHHS full resolution (scale 'f')", lambda **kw: cfeature.GSHHSFeature(scale="f", levels=[1], **kw)),
                   ("GSHHS high resolution (scale 'h')", lambda **kw: cfeature.GSHHSFeature(scale="h", levels=[1], **kw)),
                   ("Natural Earth 10m land", lambda **kw: cfeature.NaturalEarthFeature("physical", "land", "10m", **kw))]:
    try:
        probe(make(facecolor=LAND_FC, edgecolor=LAND_EC, linewidth=0.4))
        COAST = make
        print("Coastline source used:", desc)
        COAST_NAME = desc
        break
    except Exception as err:
        print("coastline source failed:", desc, "-", str(err)[:100])
if COAST is None:
    raise RuntimeError("no coastline source available")


def draw_land(ax, on_top=True):
    """on_top=True hides data on land; False draws the land fill below the data plus a coastline line above it."""
    if on_top:
        ax.add_feature(COAST(facecolor=LAND_FC, edgecolor=LAND_EC, linewidth=0.45), zorder=3)
    else:
        ax.add_feature(COAST(facecolor=LAND_FC, edgecolor="none"), zorder=0.5)
        ax.add_feature(COAST(facecolor="none", edgecolor=LAND_EC, linewidth=0.45), zorder=3)


HALO = [pe.withStroke(linewidth=2.2, foreground="white")]
RED, GREEN = "#d62728", "#2ca02c"
kind_colour = lambda k: RED if k == "Erosion" else GREEN


def draw_study_area(ax):
    for geom in study.geometry:
        ax.add_geometries([geom], crs=PC, facecolor="none", edgecolor="#8b0000", linewidth=0.9,
                          linestyle=(0, (4, 2)), zorder=6)


def label_positions(span_lat, sep_frac=0.05):
    """Label y positions: start at each zone's latitude, then push close labels apart vertically."""
    y = POS.set_index("label")["lat"].astype(float).to_dict()
    labs = sorted(y, key=lambda l: -y[l])
    sep = sep_frac * span_lat
    for _ in range(50):
        moved = False
        for a, b in zip(labs[:-1], labs[1:]):
            gap = y[a] - y[b]
            if gap < sep - 1e-9:
                shift = (sep - gap) / 2
                y[a] += shift; y[b] -= shift; moved = True
        if not moved:
            break
    return y


def draw_hotspots(ax, span_lat, label_dx=0.03, fontsize=8, marker=38, side="left"):
    """Polygon outlines (if any), filled markers at the zone points, labels on the seaward (left) side."""
    for lab, geom in ZONE_GEOMS.items():
        kind = ZONE_INFO[lab][1]
        ax.add_geometries([geom], crs=PC, facecolor="none", edgecolor=kind_colour(kind), linewidth=0.7, zorder=6)
    ys = label_positions(span_lat)
    for _, z in POS.iterrows():
        ax.scatter(z.lon, z.lat, s=marker, c=kind_colour(z.kind), edgecolors="black", linewidths=0.7,
                   zorder=8, transform=PC)
        sign = -1 if side == "left" else 1          # left = seaward; "right" only where the sea side is crowded
        ax.text(z.lon + sign * label_dx, ys[z.label], z.label, ha="right" if side == "left" else "left", va="center", fontsize=fontsize,
                fontweight="bold", zorder=9, transform=PC, path_effects=HALO)


def style_axes(ax, extent, step, left=True):
    ax.set_extent(extent, crs=PC)
    ax.set_xticks(np.arange(np.ceil(extent[0] / step) * step, extent[1] + 1e-9, step), crs=PC)
    ax.set_yticks(np.arange(np.ceil(extent[2] / step) * step, extent[3] + 1e-9, step), crs=PC)
    ax.xaxis.set_major_formatter(LongitudeFormatter(number_format=".1f"))
    ax.yaxis.set_major_formatter(LatitudeFormatter(number_format=".1f"))
    ax.gridlines(draw_labels=False, linewidth=0.3, color="gray", alpha=0.5, linestyle=":")
    ax.tick_params(labelsize=8)
    if not left:
        ax.tick_params(labelleft=False)
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.6)


def map_row(extent, step, figsize, titles):
    fig, axes = plt.subplots(1, len(titles), figsize=figsize, subplot_kw=dict(projection=PC),
                             constrained_layout=True)
    axes = np.atleast_1d(axes)
    for i, (ax, t) in enumerate(zip(axes, titles)):
        style_axes(ax, extent, step, left=(i == 0))
        ax.set_title(t, fontsize=10)
    return fig, axes


# legend handles shared by all maps
LEG_HANDLES = [Line2D([], [], color="#8b0000", ls=(0, (4, 2)), lw=1, label="Study area"),
               Line2D([], [], marker="o", ls="", mfc=RED, mec="black", ms=6, label="Erosion zone"),
               Line2D([], [], marker="o", ls="", mfc=GREEN, mec="black", ms=6, label="Accretion zone")]
SEASON_TITLES = ["SW monsoon (Jun–Sep)", "NE monsoon (Dec–Feb)", "Annual"]
SEASON_KEYS = ["sw", "ne", "ann"]

In [ ]:
# FIGURES 1 and 2: HYCOM currents and SST
HY_EXTENT = (91.3, 92.5, 20.4, 21.8)
GX, GY = np.meshgrid(np.arange(91.3, 92.5 + 1e-9, 0.01), np.arange(20.4, 21.8 + 1e-9, 0.01))


def interpolate(df, var):
    """Display only: linear interpolation 0.08 -> 0.01 deg, remaining gaps filled with nearest values."""
    pts, vals = df[["lon", "lat"]].values, df[var].values
    grid = griddata(pts, vals, (GX, GY), method="linear")
    near = griddata(pts, vals, (GX, GY), method="nearest")
    return np.where(np.isnan(grid), near, grid)


def hycom_figure(var, cmap, vmin, vmax, cbar_label, title, fname, arrows):
    fig, axes = map_row(HY_EXTENT, 0.5, (13, 6.6), SEASON_TITLES)
    span = HY_EXTENT[3] - HY_EXTENT[2]
    for ax, k in zip(axes, SEASON_KEYS):
        df = HYCOM[k]
        ax.pcolormesh(GX, GY, interpolate(df, var), cmap=cmap, vmin=vmin, vmax=vmax, shading="auto",
                      transform=PC, zorder=1, rasterized=True)
        draw_land(ax, on_top=True)
        if arrows:   # original cell centres only, every second cell in each direction
            lons, lats = np.sort(df["lon"].unique())[::2], np.sort(df["lat"].unique())[::2]
            sub = df[df["lon"].isin(lons) & df["lat"].isin(lats)]
            q = ax.quiver(sub["lon"], sub["lat"], sub["u"], sub["v"], color="white", edgecolors="0.15",
                          linewidth=0.4, scale=5.0, scale_units="width", width=0.005, headwidth=4,
                          headlength=4.5, zorder=7, transform=PC)
            last_q = q
        draw_study_area(ax)
        draw_hotspots(ax, span)
    fig.suptitle(title, fontsize=12)
    if arrows:   # one shared key (same scale in every panel), placed in the bottom row
        axes[-1].quiverkey(last_q, 0.86, 0.022, 0.2, "0.2 m s⁻¹", labelpos="E", coordinates="figure",
                           fontproperties=dict(size=8))
    cb = fig.colorbar(ScalarMappable(norm=Normalize(vmin, vmax), cmap=cmap), ax=list(axes), location="bottom",
                      shrink=0.45, aspect=40, pad=0.02, label=cbar_label)
    fig.legend(handles=LEG_HANDLES, loc="outside lower center", ncol=3, frameon=False, fontsize=8)
    fig.text(0.005, 0.005, "HYCOM 0.08° reanalysis; colours interpolated for display.", fontsize=7, color="0.3",
             ha="left", va="bottom")
    save_fig(fig, fname)


speeds = np.concatenate([HYCOM[k]["speed"].values for k in SEASON_KEYS])
hycom_figure("speed", "viridis", 0, float(np.percentile(speeds, 98)), "Surface current speed (m s⁻¹)",
             "HYCOM mean surface currents, 1993–2023", "Fig_currents_seasonal.png", arrows=True)
ssts = np.concatenate([HYCOM[k]["sst"].values for k in SEASON_KEYS])
hycom_figure("sst", "RdYlBu_r", float(np.percentile(ssts, 2)), float(np.percentile(ssts, 98)),
             "Sea surface temperature (°C)", "HYCOM mean sea surface temperature, 1993–2023",
             "Fig_SST_seasonal.png", arrows=False)

In [ ]:
# FIGURE 3: ERA5 wave height and direction
WV_EXTENT = (90.2, 92.6, 19.7, 22.0)
cmap_w = plt.get_cmap("YlGnBu")
norm_w = Normalize(WAVES["hs_mean"].min(), WAVES["hs_mean"].max())      # shared across panels

fig, axes = map_row(WV_EXTENT, 0.5, (13, 5.6), SEASON_TITLES)
for ax, k in zip(axes, SEASON_KEYS):
    draw_land(ax, on_top=False)
    for _, r in WAVE_SEASON[k].iterrows():
        ax.add_patch(Rectangle((r.grid_lon - 0.25, r.grid_lat - 0.25), 0.5, 0.5, facecolor=cmap_w(norm_w(r.hs_mean)),
                               edgecolor="0.4", linewidth=0.5, alpha=0.92, zorder=1, transform=PC))
        ax.text(r.grid_lon, r.grid_lat - 0.25 + 0.5 / 6, f"{r.hs_mean:.2f} m", ha="center", va="center",
                fontsize=8, zorder=5, transform=PC, path_effects=HALO)          # lower third
        e, n = np.sin(np.radians(r.dir_prop)), np.cos(np.radians(r.dir_prop))
        ax.quiver(r.grid_lon, r.grid_lat + 0.10, e, n, color="black", pivot="mid", scale=9, scale_units="width",
                  width=0.007, headwidth=4, zorder=5, transform=PC)             # upper half
    draw_study_area(ax)
    draw_hotspots(ax, WV_EXTENT[3] - WV_EXTENT[2], label_dx=0.04, side="right")   # labels on the land side: the sea side holds the Hs values and arrows
fig.suptitle("ERA5 mean significant wave height and wave direction, 1993–2023", fontsize=12)
fig.colorbar(ScalarMappable(norm=norm_w, cmap=cmap_w), ax=list(axes), location="bottom", shrink=0.45, aspect=40,
             pad=0.02, label="Mean significant wave height, Hs (m)")
fig.legend(handles=LEG_HANDLES + [Line2D([], [], marker=r"$\rightarrow$", ls="", color="black", ms=9,
                                         label="Wave propagation direction")],
           loc="outside lower center", ncol=4, frameon=False, fontsize=8)
save_fig(fig, "Fig_waves_seasonal.png")

In [ ]:
# FIGURE 4: hotspot forcing bars (zones north -> south)
L = LINK.set_index("label").loc[ORDER].reset_index()
fig, axes = plt.subplots(1, 3, figsize=(13, 4.8), constrained_layout=True)
panels = [("epr", "End Point Rate (m yr⁻¹)", "Shoreline change rate (EPR)"),
          ("cur_along_sw", "Alongshore current (m s⁻¹, + toward NNW)", "SW-monsoon alongshore current"),
          ("Pl_net_ann", "Longshore wave-energy-flux index (relative units)\n(+ toward NNW)",
           "Annual longshore wave-energy flux")]
for i, (ax, (col, ylabel, title)) in enumerate(zip(axes, panels)):
    colours = [kind_colour(k) for k in L["kind"]] if i == 0 else "steelblue"
    ax.bar(L["label"], L[col], color=colours, edgecolor="black", linewidth=0.6)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
fig.suptitle("Shoreline change and offshore forcing by hotspot zone (north → south)", fontsize=12)
fig.legend(handles=[Patch(fc=RED, ec="black", label="Erosion"), Patch(fc=GREEN, ec="black", label="Accretion")],
           loc="outside lower center", ncol=2, frameon=False, fontsize=8)
save_fig(fig, "Fig_hotspot_forcing.png")

In [ ]:
# FIGURES 5-7: trends and cyclones redrawn from CSV
def trend_line(x, y):
    """Sen slope + Mann-Kendall p, used only to label the trend line."""
    tau, p = stats.kendalltau(x, y)
    slope, intercept, _, _ = stats.theilslopes(y, x, 0.95)
    return slope, intercept, p


def p_text(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


# Figure 5: SST trend
sst = read_csv("sst_annual.csv")
s, b, p = trend_line(sst["year"], sst["sst"])
fig, ax = plt.subplots(figsize=(7.5, 4.3), constrained_layout=True)
ax.plot(sst["year"], sst["sst"], "o-", color="#d62728", ms=4, lw=1, label="Annual mean SST")
ax.plot(sst["year"], b + s * sst["year"], "k--", lw=1.4, label=f"Sen slope {s * 10:+.2f} °C per decade ({p_text(p)})")
ax.set_xlabel("Year"); ax.set_ylabel("Sea surface temperature (°C)")
ax.set_title("HYCOM annual mean SST, nearshore box (91.8–92.4°E, 20.6–21.6°N)")
ax.grid(alpha=0.3); ax.legend(frameon=False)
save_fig(fig, "Fig_SST_trend.png")

# Figure 6: wave climatology and trend
hm, ha = read_csv("hs_monthly.csv"), read_csv("hs_annual.csv")
s, b, p = trend_line(ha["year"], ha["hs"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.3), constrained_layout=True)
a1.bar(hm["month"], hm["hs"], color="steelblue", edgecolor="black", linewidth=0.5)
a1.set_xticks(range(1, 13)); a1.set_xticklabels(list("JFMAMJJASOND"))
a1.set_xlabel("Month"); a1.set_ylabel("Mean significant wave height, Hs (m)")
a1.set_title("Monthly mean Hs"); a1.grid(axis="y", alpha=0.3); a1.set_axisbelow(True)
a2.plot(ha["year"], ha["hs"], "o-", color="steelblue", ms=4, lw=1, label="Annual mean Hs")
a2.plot(ha["year"], b + s * ha["year"], "k--", lw=1.4, label=f"Sen slope {s * 10:+.3f} m per decade ({p_text(p)})")
a2.set_xlabel("Year"); a2.set_ylabel("Mean significant wave height, Hs (m)")
a2.set_title("Annual mean Hs, all ERA5 cells pooled"); a2.grid(alpha=0.3); a2.legend(frameon=False)
save_fig(fig, "Fig_waves_climatology_trend.png")

# Figure 7: cyclones per 5-year period
cy = read_csv("cyclones_250km.csv")
starts = list(range(1992, 2024, 5))
counts = [int(((cy["season"] >= s0) & (cy["season"] <= min(s0 + 4, 2023))).sum()) for s0 in starts]
fig, ax = plt.subplots(figsize=(7.5, 4.3), constrained_layout=True)
bars = ax.bar([f"{s0}–{min(s0 + 4, 2023)}" for s0 in starts], counts, color="slategray", edgecolor="black", linewidth=0.5)
ax.bar_label(bars, fontsize=8)
ax.set_xlabel("Period"); ax.set_ylabel("Number of storms")
ax.set_title("Cyclones passing within 250 km of the Cox's Bazar–Teknaf coast")
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True); ax.margins(y=0.12)
save_fig(fig, "Fig_cyclones.png")

In [ ]:
# FIGURE 8: study area and hotspot zones
x0, y0, x1, y1 = study.total_bounds
EXT8 = (x0 - 0.1, x1 + 0.1, y0 - 0.1, y1 + 0.1)
fig = plt.figure(figsize=(6.2, 8.2), constrained_layout=True)
ax = fig.add_subplot(projection=PC)
style_axes(ax, EXT8, 0.2)
ax.set_facecolor("#dbeaf4")                     # sea
draw_land(ax, on_top=True)
for geom in study.geometry:                     # light fill so the outline reads as an area
    ax.add_geometries([geom], crs=PC, facecolor="#8b0000", alpha=0.07, edgecolor="none", zorder=4)
draw_study_area(ax)
draw_hotspots(ax, EXT8[3] - EXT8[2], label_dx=0.025, fontsize=10, marker=60)
ax.set_title("Study area and hotspot zones", fontsize=12)

# north arrow (axes coordinates)
ax.annotate("", xy=(0.93, 0.95), xytext=(0.93, 0.86), xycoords="axes fraction",
            arrowprops=dict(arrowstyle="-|>", color="black", lw=1.5), zorder=10)
ax.text(0.93, 0.96, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=11, fontweight="bold", zorder=10)

# scale bar: 20 km at the lower left (open sea) (length converted to degrees of longitude at that latitude)
lat_sb = EXT8[2] + 0.07
deg20 = 20 / (111.32 * np.cos(np.radians(lat_sb)))
lon_end = EXT8[0] + 0.04 + deg20
deg20 = 20 / (111.32 * np.cos(np.radians(lat_sb)))
ax.plot([lon_end - deg20, lon_end], [lat_sb, lat_sb], color="black", lw=3, transform=PC, zorder=10,
        solid_capstyle="butt")
ax.plot([lon_end - deg20 / 2, lon_end], [lat_sb, lat_sb], color="white", lw=1.5, transform=PC, zorder=11,
        solid_capstyle="butt")
for frac, txt in ((0, "0"), (0.5, "10"), (1, "20 km")):
    ax.text(lon_end - deg20 + frac * deg20, lat_sb + 0.025, txt, ha="center", va="bottom", fontsize=8,
            transform=PC, zorder=10, path_effects=HALO)
fig.legend(handles=LEG_HANDLES, loc="outside lower center", ncol=3, frameon=False, fontsize=8)
save_fig(fig, "Fig_study_area_hotspots.png")

In [ ]:
# SUMMARY v2 and download
def clean(df, nd=3):
    """Rounded records with NaN -> None so the JSON is valid."""
    out = df.copy()
    num = out.select_dtypes("number").columns
    out[num] = out[num].round(nd)
    return json.loads(out.to_json(orient="records"))


summary = {"coastline_source": COAST_NAME, "hotspot_match_method": MATCH_METHOD,
           "hotspot_positions": clean(POS[["zone", "label", "kind", "epr", "lat", "lon", "shift_km"]], 5),
           "linkage_v2": clean(LINK), "correlations_v2": clean(CORR),
           "n_values_changed_vs_original": int(len(CHANGES))}
with open(os.path.join(FIG_DIR, "SUMMARY_v2.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
print("==== SUMMARY v2 (send to Claude) ====")
print(json.dumps(summary, indent=2, ensure_ascii=False))

shutil.make_archive("figures_v2", "zip", FIG_DIR)
try:
    from google.colab import files
    files.download("figures_v2.zip")
except Exception as err:
    print("Automatic download not available:", err)
    print("The zip is at", os.path.abspath("figures_v2.zip"))